In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# Handling

num_cols = df.select_dtypes(include=["int64","float64"]).columns
for i in num_cols:
  mean = df[i].mean()
  df[i] = df[i].fillna(mean)
check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
cat_cols = df.select_dtypes(include='object').columns
display(cat_cols)

#No encoding needed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()




In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

#Yes there is a great imbalance of the target values

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

#We have to use Stratified K fold splitting
all_results = {}
all_results['CatBoost'] = {'accuracy': [], 'f1': []}

n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

#Creation of catboost model
CB =  CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training {'Cat Boost'}...")
  CB.fit(X_train, y_train) # train
  y_pred =  CB.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results['CatBoost']['accuracy'].append(accuracy)
  all_results['CatBoost']['f1'].append(f1)


print(f"  Accuracy:  {np.mean(all_results['CatBoost']['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(all_results['CatBoost']['f1']):.4f}")
#Averaging score
avg_f1 = np.mean(all_results['CatBoost']['f1'], axis=0)
print("")
print(f"Average f1 score {avg_f1}") #By score I am assuming f1 score

In [ ]:
# Task 1: Write your code here:
importances = {}
importances['CatBoost'] = CB.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 1, figsize=(18, 6))

features = X.columns
for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot

  sorted_idx = np.argsort(imp)
  ax = axes
  ax.barh(features[sorted_idx], imp[sorted_idx])

  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: